# Phase 3 — Unsupervised Anomaly Detection on UNSW-NB15

**Models implemented:**
1. **Isolation Forest** — an ensemble of random isolation trees
2. **Dense Feedforward Autoencoder** — a PyTorch reconstruction-error model

Both models are trained **only on normal traffic** and evaluated on the full labelled test set.

---

## Why Unsupervised Anomaly Detection?

> In real-world SIEM deployments, **labelled attack traffic is scarce and perpetually outdated**. New zero-day exploits produce traffic patterns never seen in training data. Supervised classifiers (RF, XGBoost) can only detect attacks that resemble training examples.
>
> Unsupervised methods learn a model of **what normal looks like**. Any traffic that deviates significantly from that model is flagged — regardless of whether it matches a known attack signature. This is the foundational principle behind anomaly-based intrusion detection.

## Why Train Only on Normal Traffic?

> Training on a mixture of normal and attack traffic would teach the model what attacks look like too, erasing the anomaly signal. By training only on normal samples, the learned representation (IF trees / AE bottleneck) captures the manifold of genuine benign behaviour. Attack flows lie off this manifold and produce anomalous scores.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.isolation_forest import IsolationForestDetector
from src.models.autoencoder import DenseAutoencoderDetector

# ── Paths ─────────────────────────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR    = PROJECT_ROOT / "models"
REPORTS_DIR   = PROJECT_ROOT / "reports"

IF_PATH   = MODELS_DIR / "isolation_forest.joblib"
AE_PATH   = MODELS_DIR / "autoencoder.pt"
EVAL_PATH = REPORTS_DIR / "unsupervised_evaluation.json"
FEAT_PATH = PROCESSED_DIR / "feature_list.txt"

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
PALETTE = {"normal": "#4C72B0", "attack": "#DD4444"}

print("Imports OK")

## 1. Load Trained Models

In [ ]:
ifd = IsolationForestDetector.load_model(IF_PATH)
ae  = DenseAutoencoderDetector.load_model(AE_PATH)

print(f"Isolation Forest threshold : {ifd._threshold:.6f}")
print(f"Autoencoder threshold      : {ae._threshold:.6f}")
print(f"Autoencoder input_dim      : {ae._input_dim}")

## 2. Load Evaluation Report

In [ ]:
with open(EVAL_PATH) as f:
    report = json.load(f)

if_m = report["models"]["isolation_forest"]
ae_m = report["models"]["autoencoder"]

meta = report["metadata"]
print(f"Normal train rows : {meta['normal_train_rows']:,}")
print(f"Test rows         : {meta['test_rows']:,}")
print(f"  Attack          : {meta['test_attack_rows']:,}")
print(f"  Normal          : {meta['test_normal_rows']:,}")

## 3. Load Test Data & Compute Scores

In [ ]:
feature_names = FEAT_PATH.read_text().strip().splitlines()

test_df = pd.read_parquet(
    PROCESSED_DIR / "test.parquet",
    columns=feature_names + ["label"]
)
X_test  = test_df[feature_names].values.astype(np.float32)
y_test  = (test_df["label"] == 1).astype(int).values

print(f"Test shape: {X_test.shape}")
print(f"Attack fraction: {y_test.mean():.3%}")

# ── Compute anomaly scores ─────────────────────────────────────────────────
print("\nComputing IF scores…")
if_scores = -ifd._model.score_samples(X_test)          # higher = more anomalous

print("Computing AE reconstruction errors…")
ae_errors = ae._reconstruction_errors(X_test)           # higher = more anomalous

print("Done.")

## 4. Anomaly Score Distributions

A well-calibrated anomaly detector should produce **clearly separated** score distributions for normal vs attack traffic. The vertical dashed line is the trained threshold.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# ── Isolation Forest ──────────────────────────────────────────────────────────
ax = axes[0]
ax.hist(
    if_scores[y_test == 0], bins=120, alpha=0.65,
    color=PALETTE["normal"], label="Normal", density=True
)
ax.hist(
    if_scores[y_test == 1], bins=120, alpha=0.65,
    color=PALETTE["attack"], label="Attack", density=True
)
ax.axvline(ifd._threshold, color="black", ls="--", lw=1.5,
           label=f"Threshold={ifd._threshold:.4f}")
ax.set_title("Isolation Forest — Anomaly Score Distribution")
ax.set_xlabel("Anomaly Score  (higher = more anomalous)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)

# ── Autoencoder ───────────────────────────────────────────────────────────────
ax = axes[1]
ax.hist(
    ae_errors[y_test == 0], bins=120, alpha=0.65,
    color=PALETTE["normal"], label="Normal", density=True
)
ax.hist(
    ae_errors[y_test == 1], bins=120, alpha=0.65,
    color=PALETTE["attack"], label="Attack", density=True
)
ax.axvline(ae._threshold, color="black", ls="--", lw=1.5,
           label=f"Threshold={ae._threshold:.4f}")
ax.set_title("Dense Autoencoder — Reconstruction Error Distribution")
ax.set_xlabel("Reconstruction Error (MSE)  (higher = more anomalous)")
ax.set_ylabel("Density")
ax.legend(fontsize=9)

fig.suptitle("Anomaly Score Distributions: Normal vs Attack Traffic",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "unsupervised_score_distributions.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 5. ROC Curves

ROC-AUC measures discrimination ability across ALL possible thresholds. A value of 1.0 = perfect; 0.5 = random.

**How anomaly scores generate ROC curves:**
- Isolation Forest: `anomaly_score = -score_samples()`. Higher score → more likely attack.
- Autoencoder: `anomaly_score = MSE(input, reconstruction)`. Higher error → more likely attack.
- By varying the threshold from min to max, we sweep through FPR/TPR space to form the ROC curve.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for scores, label, color in [
    (if_scores, "Isolation Forest",    "#2176AE"),
    (ae_errors, "Dense Autoencoder",   "#E05C2E"),
]:
    fpr_arr, tpr_arr, _ = roc_curve(y_test, scores)
    roc_auc_val = auc(fpr_arr, tpr_arr)
    ax.plot(fpr_arr, tpr_arr, lw=2, color=color,
            label=f"{label}  (AUC = {roc_auc_val:.4f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.50)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Unsupervised Anomaly Detectors", fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(REPORTS_DIR / "unsupervised_roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Precision / Recall / F1 Comparison

These metrics are computed at the trained threshold (p95 of training normal scores).

### Why thresholds affect false positives and false negatives

> - **Lower threshold** → more samples flagged → higher Recall (fewer missed attacks) but higher FPR (more false alarms on normal traffic).
> - **Higher threshold** → fewer samples flagged → lower FPR but lower Recall (more missed attacks).
> - The **p95 threshold** represents a deliberate operating point: we accept that 5% of training normal traffic is considered "borderline anomalous" in order to detect attacks with a reasonable threshold margin.

In [ ]:
metrics_to_plot = ["precision", "recall", "f1", "roc_auc", "false_positive_rate"]
labels_map = {
    "precision": "Precision",
    "recall": "Recall",
    "f1": "F1 Score",
    "roc_auc": "ROC-AUC",
    "false_positive_rate": "False Positive Rate",
}

if_vals  = [if_m.get(m, 0) or 0 for m in metrics_to_plot]
ae_vals  = [ae_m.get(m, 0) or 0 for m in metrics_to_plot]
x        = np.arange(len(metrics_to_plot))
width    = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, if_vals, width, label="Isolation Forest",
               color="#2176AE", alpha=0.85)
bars2 = ax.bar(x + width/2, ae_vals, width, label="Dense Autoencoder",
               color="#E05C2E", alpha=0.85)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.008,
            f"{h:.3f}", ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels([labels_map[m] for m in metrics_to_plot], fontsize=10)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Score")
ax.set_title("Performance Comparison at p95 Threshold", fontweight="bold")
ax.legend(fontsize=10)
ax.axhline(0.5, color="grey", ls=":", lw=1)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "unsupervised_metrics_comparison.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, metrics, title in [
    (axes[0], if_m, "Isolation Forest"),
    (axes[1], ae_m, "Dense Autoencoder"),
]:
    cm = np.array(metrics["confusion_matrix"])
    sns.heatmap(
        cm, annot=True, fmt=",d", ax=ax,
        cmap="Blues", cbar=False,
        xticklabels=["Pred Normal", "Pred Attack"],
        yticklabels=["True Normal", "True Attack"],
    )
    ax.set_title(f"{title}\nConfusion Matrix", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(REPORTS_DIR / "unsupervised_confusion_matrices.png",
            dpi=150, bbox_inches="tight")
plt.show()

## 8. Autoencoder Training History

In [ ]:
history = ae_m.get("train_history", [])
if history:
    epochs_ax = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss   = [h["val_loss"]   for h in history]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs_ax, train_loss, lw=2, color="#2176AE", label="Train Loss (MSE)")
    ax.plot(epochs_ax, val_loss,   lw=2, color="#E05C2E",
            ls="--", label="Val Loss (MSE)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.set_title("Autoencoder Training History", fontweight="bold")
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / "ae_training_history.png",
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No training history found in report.")

## 9. Summary Table

In [ ]:
rows = []
for name, m in [("Isolation Forest", if_m), ("Dense Autoencoder", ae_m)]:
    rows.append({
        "Model": name,
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
        "ROC-AUC": m["roc_auc"],
        "FPR": m["false_positive_rate"],
        "Threshold": m["threshold"],
        "TP": m["tp"],
        "FP": m["fp"],
        "TN": m["tn"],
        "FN": m["fn"],
    })

summary_df = pd.DataFrame(rows).set_index("Model")
summary_df

## 10. Discussion — Strengths & Weaknesses

### Why Dense Autoencoder instead of LSTM?

UNSW-NB15 is **tabular network flow data** — each row is an independently aggregated bidirectional flow record with engineered features (duration, bytes, packet counts, etc.). There is no meaningful temporal ordering between rows in the stored dataset.

LSTMs are designed for sequential data where the **order** of observations encodes information (e.g., raw byte streams, log event chains). Using an LSTM here would require:
1. Artificially constructing windows of consecutive rows — imposing a fake temporal structure.
2. Choosing an arbitrary window length with no principled basis.
3. Significantly higher training time with no accuracy benefit.

A Dense Autoencoder treats each flow record as a fixed-size feature vector, learning a compact bottleneck (8-dim) representation of normal behaviour. This is the architecturally correct choice.

---

### Isolation Forest

| Strength | Weakness |
|---|---|
| No distributional assumptions | Cannot capture complex feature interactions |
| Scales to millions of rows | Threshold is heuristic (percentile-based) |
| Fast training (parallel trees) | Susceptible to masking in high dimensions |
| No gradient/optimisation needed | No incremental learning |

### Dense Autoencoder

| Strength | Weakness |
|---|---|
| Learns non-linear feature interactions | Longer training time than IF |
| Bottleneck forces compact normal representation | Sensitive to learning rate & architecture |
| Reconstruction error is interpretable | May reconstruct some attacks well if they're close to normal manifold |
| Early stopping prevents overfitting | Requires GPU for best speed |

### Threshold Sensitivity

Both models use a **p95 threshold**:
- Lowering to p90 → higher recall, higher FPR (more false alarms).
- Raising to p99 → lower FPR, lower recall (more missed attacks).
- The threshold is a **business decision**: for a SIEM, analyst capacity constraints dictate the maximum tolerable FPR.